# Reddibase — Train a new identification model

Trains a solved classifier and retrieval model for any subreddit with a
"vague description → confirmed answer" structure (e.g. r/tipofmyjoystick,
r/tipofmytongue, r/whatsthisbug).

**Before you start:**
1. Enable GPU: Runtime → Change runtime type → T4 GPU
2. Mount Google Drive (cell below) — scraping takes hours and Drive keeps
   your progress if the session disconnects
3. Create a [Hugging Face account](https://huggingface.co) and generate a
   write-access token at huggingface.co/settings/tokens

**Estimated time on a free T4:**
- Scraping: 2–8 hours (run overnight, restartable)
- Classifier training: 2–4 hours
- Embedder + index: 30–60 minutes

## 1 · Mount Drive and install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
repo_url = 'https://github.com/eftpmc/reddibase' #@param {type:"string"}

import subprocess, sys
subprocess.run(['git', 'clone', '-q', repo_url, '/content/reddibase'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                '/content/reddibase/requirements.txt'], check=True)

sys.path.insert(0, '/content/reddibase')
print('Ready.')

## 2 · Configuration

Fill in the form fields, then run the cell.

In [ ]:
subreddit        = 'tipofmyjoystick' #@param {type:"string"}
hf_repo          = 'eftpmc/tipofmyjoystick-identification' #@param {type:"string"}
clf_hf_repo      = 'eftpmc/reddibase-solved-classifier' #@param {type:"string"}
hf_token         = '' #@param {type:"string"}
classifier_repo  = '' #@param {type:"string"}
# ^ Leave empty to train a new classifier. Set to clf_hf_repo to reuse an existing one.

# 'any_flair_is_solved': any flair = solved, flair text = answer (tipofmyjoystick)
# 'specific_flairs': only posts whose flair matches one of solved_flairs below
flair_strategy = 'any_flair_is_solved' #@param ["any_flair_is_solved", "specific_flairs"]
solved_flairs  = 'Solved!, solved' #@param {type:"string"}

CONFIDENCE_THRESHOLD = 0.80 #@param {type:"slider", min:0.5, max:0.99, step:0.01}

import os

DRIVE_DIR   = f'/content/drive/MyDrive/reddibase/{subreddit}'
POSTS_CACHE = f'{DRIVE_DIR}/posts.pkl'
PAIRS_FILE  = f'{DRIVE_DIR}/confirmed_pairs.jsonl'
CLF_DIR     = f'{DRIVE_DIR}/solved_classifier'
MODEL_DIR   = f'{DRIVE_DIR}/identification_model'

os.makedirs(DRIVE_DIR, exist_ok=True)

print(f'Subreddit       : {subreddit}')
print(f'Drive dir       : {DRIVE_DIR}')
print(f'Classifier dir  : {CLF_DIR}')
print(f'Model dir       : {MODEL_DIR}')
print()
if classifier_repo:
    print(f'Classifier HF   : {classifier_repo}  (reusing — skip 7a and 7b)')
else:
    print(f'Classifier HF   : {clf_hf_repo}  (will train + push)')
print(f'Identifier HF   : {hf_repo}')

## 2.5 · Verify API before scraping

Fetches 3 posts and 3 comments from Arctic Shift and prints the raw fields.
**Check that the output looks sensible before starting the full scrape.**
If you see an error or empty data, the API endpoint may have changed — open
an issue on the Reddibase repo with the error text.

In [ ]:
import httpx

BASE = 'https://arctic-shift.photon-reddit.com/api'

with httpx.Client(timeout=15) as client:
    # --- posts ---
    r = client.get(f'{BASE}/posts/search',
                   params={'subreddit': subreddit, 'limit': 3,
                           'sort': 'asc', 'sort_type': 'created_utc'})
    r.raise_for_status()
    posts_sample = r.json().get('data', [])
    print(f'Posts endpoint OK — {len(posts_sample)} returned')
    if posts_sample:
        p = posts_sample[0]
        print(f'  Keys       : {list(p.keys())}')
        print(f'  id         : {p.get("id")}')
        print(f'  title      : {p.get("title", "")[:70]!r}')
        print(f'  flair      : {p.get("link_flair_text")!r}')
        print(f'  author     : {p.get("author")}')
        print(f'  created_utc: {p.get("created_utc")}')
        print(f'  score      : {p.get("score")}')

    # --- comments ---
    if posts_sample:
        sample_id = posts_sample[0]['id']
        r2 = client.get(f'{BASE}/comments/search',
                        params={'link_id': f't3_{sample_id}', 'limit': 3})
        r2.raise_for_status()
        comments_sample = r2.json().get('data', [])
        print(f'\nComments endpoint OK — {len(comments_sample)} returned for post {sample_id}')
        if comments_sample:
            c = comments_sample[0]
            print(f'  Keys     : {list(c.keys())}')
            print(f'  id       : {c.get("id")}')
            print(f'  author   : {c.get("author")}')
            print(f'  parent_id: {c.get("parent_id")}')
            print(f'  score    : {c.get("score")}')
            print(f'  body     : {c.get("body", "")[:80]!r}')

print('\nIf the fields above look right, proceed to Section 3.')

## 3 · Scrape all posts

Pulls every post and full comment thread from the Arctic Shift archive.
Progress is checkpointed to Drive every 1000 posts — if your session
disconnects, re-run this cell and it will pick up where it stopped.

In [ ]:
import asyncio, pickle
from pathlib import Path
from tqdm.notebook import tqdm
from framework.scraper import ArcticShiftScraper

CHECKPOINT_EVERY = 1000

if Path(POSTS_CACHE).exists():
    with open(POSTS_CACHE, 'rb') as f:
        posts = pickle.load(f)
    resume_after = posts[-1].created_utc + 1
    print(f'Resuming: {len(posts):,} posts already collected')
else:
    posts = []
    resume_after = 0
    print('Starting fresh scrape')

async def scrape():
    new_count = 0
    async with ArcticShiftScraper(subreddit, after=resume_after, request_delay=1.0) as scraper:
        async for post in scraper.stream_posts():
            posts.append(post)
            new_count += 1
            if new_count % CHECKPOINT_EVERY == 0:
                with open(POSTS_CACHE, 'wb') as f:
                    pickle.dump(posts, f)
                print(f'  Checkpoint: {len(posts):,} total posts')
    with open(POSTS_CACHE, 'wb') as f:
        pickle.dump(posts, f)

await scrape()

flaired = [p for p in posts if p.flair]
print(f'Done. {len(posts):,} posts, {len(flaired):,} flaired ({len(flaired)/len(posts)*100:.1f}%)')

## 4 · Train the solved classifier

Fine-tunes DistilBERT on bootstrap labels from flaired posts.
The T4 has 16 GB VRAM so we use a larger batch than the conservative
local defaults.

In [ ]:
import os, pickle
from framework.classifier import BootstrapDataBuilder, SolvedClassifier

if classifier_repo:
    from huggingface_hub import snapshot_download
    import shutil
    print(f'Downloading classifier from {classifier_repo}...')
    cached = snapshot_download(repo_id=classifier_repo)
    if os.path.exists(CLF_DIR):
        shutil.rmtree(CLF_DIR)
    shutil.copytree(cached, CLF_DIR)
    print(f'Classifier ready at {CLF_DIR}')
else:
    with open(POSTS_CACHE, 'rb') as f:
        posts = pickle.load(f)

    if flair_strategy == 'any_flair_is_solved':
        training_posts = [p for p in posts if p.flair]
    else:
        wanted = {s.strip() for s in solved_flairs.split(',')}
        training_posts = [p for p in posts if p.flair in wanted]

    print(f'Bootstrap posts: {len(training_posts):,}')

    examples = BootstrapDataBuilder().build(training_posts)
    pos = sum(1 for e in examples if e.label == 1)
    print(f'Examples: {len(examples):,}  ({pos:,} positive, {len(examples)-pos:,} negative)')

    SolvedClassifier.train(
        examples=examples,
        output_path=CLF_DIR,
        base_model='distilbert-base-uncased',
        num_epochs=3,
        batch_size=16,
        gradient_accumulation_steps=1,
    )
    print(f'Classifier saved to {CLF_DIR}')


## 5 · Build the confirmed-pairs dataset

Runs the classifier over **all** posts, including unflaired ones, to recover
solved posts OP never flaired. Only pairs above the confidence threshold are kept.

In [ ]:
import dataclasses, json, pickle
from tqdm.notebook import tqdm
from framework.classifier import SolvedClassifier

with open(POSTS_CACHE, 'rb') as f:
    posts = pickle.load(f)

clf = SolvedClassifier(CLF_DIR).load()

confirmed = []
for post in tqdm(posts, desc='Classifying'):
    is_solved, answer, confidence = clf.predict(post)
    if is_solved and confidence >= CONFIDENCE_THRESHOLD:
        confirmed.append(clf.to_confirmed_pair(post, answer, confidence))

print(f'Confirmed: {len(confirmed):,} / {len(posts):,}')

with open(PAIRS_FILE, 'w', encoding='utf-8') as f:
    for pair in confirmed:
        f.write(json.dumps(dataclasses.asdict(pair)) + '\n')

print(f'Saved to {PAIRS_FILE}')

## 6 · Train the identification model + build FAISS index

Fine-tunes a Sentence Transformer so descriptions of the same answer cluster
together, then encodes all confirmed descriptions into the static FAISS index.

In [ ]:
import json
from framework.embedder import IdentificationModel
from framework.schema import ConfirmedPair

pairs = []
with open(PAIRS_FILE, encoding='utf-8') as f:
    for line in f:
        pairs.append(ConfirmedPair(**json.loads(line)))

unique = len({(p.flair or p.answer).lower() for p in pairs})
print(f'Pairs: {len(pairs):,}  unique answers: {unique:,}')

IdentificationModel.train(
    pairs=pairs,
    base_model='sentence-transformers/all-MiniLM-L6-v2',
    output_path=MODEL_DIR,
    num_epochs=3,
    batch_size=64,
)
print(f'Model + index saved to {MODEL_DIR}')

## 7a — Create classifier repo

Skip this cell if `classifier_repo` is set (reusing an existing one).

In [ ]:
from huggingface_hub import HfApi, login

login(token=hf_token)
api = HfApi()
api.create_repo(repo_id=clf_hf_repo, repo_type='model', exist_ok=True)
print(f'Repo ready: https://huggingface.co/{clf_hf_repo}')

## 7b — Push classifier

Skip this cell if `classifier_repo` is set (reusing an existing one).

In [ ]:
api.upload_folder(folder_path=CLF_DIR, repo_id=clf_hf_repo, repo_type='model')
print(f'Classifier live at https://huggingface.co/{clf_hf_repo}')

## 7c — Create identification model repo

In [ ]:
from huggingface_hub import HfApi, login

login(token=hf_token)
api = HfApi()
api.create_repo(repo_id=hf_repo, repo_type='model', exist_ok=True)
print(f'Repo ready: https://huggingface.co/{hf_repo}')

## 7d — Upload identification model

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer(MODEL_DIR)
encoder.save(MODEL_DIR)

api.upload_folder(
    folder_path=MODEL_DIR,
    repo_id=hf_repo,
    repo_type='model',
)
print(f'Model live at https://huggingface.co/{hf_repo}')

## 8 · Generate config and submit to the registry

Copy the output below into `models/{subreddit}/config.yaml` and open a pull
request to the Reddibase repo. That's it — your model will appear in the
registry once the PR is merged.

In [ ]:
import json
from framework.schema import ConfirmedPair

pairs = []
with open(PAIRS_FILE, encoding='utf-8') as f:
    for line in f:
        pairs.append(ConfirmedPair(**json.loads(line)))

lines = [
    f'name: {subreddit}',
    f'display_name: {subreddit}  # update with a human-friendly name',
    'description: ""  # one sentence describing what this model identifies',
    f'subreddit: {subreddit}',
    f'hf_repo: {hf_repo}',
    f'classifier_repo: {classifier_repo if classifier_repo else clf_hf_repo}',
    '',
    'scraper:',
    '  after: 0',
    '  request_delay: 1.0',
    '',
    'classifier:',
    '  base_model: distilbert-base-uncased',
    f'  flair_strategy: {flair_strategy}',
    f'  confidence_threshold: {CONFIDENCE_THRESHOLD}',
    '',
    'identification_model:',
    '  base_encoder: sentence-transformers/all-MiniLM-L6-v2',
    '  top_k: 10',
    '',
    'stats:',
    f'  confirmed_pairs: {len(pairs)}',
]
print('
'.join(lines))
print()
print(f'Save this as models/{subreddit}/config.yaml and open a PR.')